# Parte 1 — Engenharia de Dados: Preparação do Lakehouse

Define os caminhos do projeto, cria catálogo, schemas e Volume e confere a presença dos cinco CSVs de entrada.


## 1. Configuração do catálogo, schemas e Landing Zone

Define os nomes usados nas camadas Bronze, Silver e Gold. O caminho do Volume indica onde os arquivos de entrada devem ser carregados.


In [0]:
# Nomes e caminhos compartilhados pelos notebooks do projeto.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: cinedata_medallion
bronze_schema: cinedata_medallion.bronze
silver_schema: cinedata_medallion.silver
gold_schema: cinedata_medallion.gold
landing_path: /Volumes/cinedata_medallion/bronze/Landing


## 2. Criação do catálogo, schemas e Volume

Cria a estrutura do Lakehouse. IF NOT EXISTS permite manter os objetos que já foram criados.


In [0]:
# Prepara a estrutura que receberá os arquivos e as tabelas das três camadas.

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")
          
print("Catalogo e esquemas prontos")

Catalogo e esquemas prontos


## 3. Conferência dos arquivos CSV no Volume

Lista os arquivos da Landing Zone e informa quais dos cinco CSVs esperados ainda estão ausentes. O upload dos arquivos é feito pelo Volume.


In [0]:
# Confere se os arquivos necessários para a carga Bronze estão disponíveis.

expected_files = {
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv",
}

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"Não foi possível listar '{landing_path}', faça o upload dos arquivos.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos aindo nao encontrados na landing zone:")
    for f in missing:
        print(f" - {f}")
else:
    print("[OK] Todos os arquivos esperados estao na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f" - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estao na landing zone:
 - cotacao_dolar.json (0.5 KB)
 - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
 - movies_financials_IMDB_TMDB.csv (1433.3 KB)
 - movies_info_TMDB_IMDB.csv (33129.0 KB)
 - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
 - movies_reviews.csv (1878.4 KB)
